# Shape Analysis: Aligning Two Configurations

**Shape analysis** studies geometric properties of point sets that are invariant under a chosen group of transformations — typically *rigid* (rotation + translation) or *similarity* (rigid + uniform scale). The recurring computational problem is **alignment**: given two configurations $A$ and $B$ of the same object, find the transformation that maps $A$ onto $B$ while minimising the sum of squared distances.

Every algorithm below solves a variant of the same least-squares problem. They differ in:

1. **What is known** — are the point correspondences given, or do we have to discover them?
2. **What is allowed** — rotation only, rotation + translation, or also a scale factor?
3. **How it is solved** — closed-form via SVD, an eigenvalue problem, or alternating iterations.

A summary table at the end places them side-by-side.

## 1. Orthogonal Procrustes

**Problem.** Given $A, B \in \mathbb{R}^{d \times n}$ (each column is a point; correspondence is by column index), find

$$
R^{\star} \;=\; \arg\min_{R \in O(d)} \; \lVert B - R\,A \rVert_F^2.
$$

**Solution.** Minimising $\lVert B - RA \rVert_F^2$ is equivalent to maximising $\operatorname{tr}(R^\top B A^\top)$. Take $M = B A^\top$, compute its SVD $M = U \Sigma V^\top$, and set

$$
\boxed{\; R^{\star} = U V^\top. \;}
$$

To restrict the answer to a *rotation* (i.e. $\det R = +1$, no reflection), correct the last singular value:

$$
R^{\star} = U \, \mathrm{diag}\!\bigl(1, \ldots, 1, \det(U V^\top)\bigr) \, V^\top.
$$

**Where this shows up — re-orthonormalising a drifted rotation matrix.** Inside an EKF or a hand-rolled pose integrator, $R$ is propagated thousands of times per second by $R \leftarrow R\,\Delta R$. Floating-point round-off pushes $R$ slowly off $SO(3)$ — its singular values drift from 1 and $\det R$ drifts from $+1$. **Project back with Procrustes** by setting $B = R,\ A = I$: SVD of $R = U\Sigma V^\top$, then $\hat R = U\,\mathrm{diag}(1, 1, \det(UV^\top))\,V^\top$ is the closest valid rotation matrix. The same one-liner repairs a 3×3 rotation regressed by a neural network (which has no orthogonality constraint by construction) — used in DeepVO, the rotation head of PoseNet, and most learning-based VO systems.

In [ ]:
import numpy as np

def orthogonal_procrustes(A, B, proper=True):
    """Return R minimising ‖B - R A‖_F. If proper=True, restrict to SO(d)."""
    M = B @ A.T
    U, _, Vt = np.linalg.svd(M)
    if proper:
        D = np.eye(M.shape[0])
        D[-1, -1] = np.sign(np.linalg.det(U @ Vt))
        return U @ D @ Vt
    return U @ Vt

rng = np.random.default_rng(0)
A = rng.standard_normal((3, 8))
R_true, _ = np.linalg.qr(rng.standard_normal((3, 3)))
if np.linalg.det(R_true) < 0:
    R_true[:, 0] *= -1
B = R_true @ A
R_hat = orthogonal_procrustes(A, B)
print("Procrustes  ‖R̂ - R‖ =", np.linalg.norm(R_hat - R_true))

## 2. Wahba's problem

**Problem.** Given $N \ge 2$ pairs of unit vectors $\{(\mathbf{a}_i, \mathbf{b}_i)\}_{i=1}^N$ measured in two frames, with non-negative weights $w_i$, find $R \in SO(3)$ minimising

$$
J(R) \;=\; \tfrac12 \sum_{i=1}^{N} w_i \, \lVert \mathbf{b}_i - R\,\mathbf{a}_i \rVert^2.
$$

Standard setting in spacecraft attitude determination: $\mathbf{a}_i$ are reference directions (sun, magnetic field, star catalog) in the inertial frame; $\mathbf{b}_i$ are the same directions measured in the body frame.

**Solution (SVD form).** Form the *attitude profile matrix*

$$
B \;=\; \sum_{i=1}^{N} w_i \, \mathbf{b}_i \mathbf{a}_i^\top \;\in\; \mathbb{R}^{3\times 3},
$$

compute $B = U\Sigma V^\top$, and set

$$
\boxed{\; R^{\star} \;=\; U \, \mathrm{diag}\!\bigl(1, 1, \det(U)\det(V)\bigr) \, V^\top. \;}
$$

The $\det$ correction enforces $\det R^{\star} = +1$. Without it, SVD might return a reflection that achieves the same Frobenius cost.

**Where this shows up — bootstrap attitude for a UAV / quadcopter on the ground.** Before take-off, the IMU is at rest. The accelerometer reads gravity $\mathbf{a}_1^{\,W} = (0, 0, -g)$ in the world frame, the magnetometer reads the local Earth field $\mathbf{a}_2^{\,W}$ (from the WMM model and the GPS-derived location). The same two physical directions, *measured* in the body frame, give $\mathbf{b}_1, \mathbf{b}_2$. Solving Wahba's problem with $N=2$ returns the world-to-body rotation — the **initial attitude** the EKF needs at boot. PX4 and ArduPilot run this exact step (the "MAG initialised" pre-arm log line) before the gyro starts integrating; the same method initialises CubeSats from sun-sensor + magnetometer measurements once on orbit.

In [ ]:
def wahba_svd(a, b, w=None):
    """a, b: (N, 3) unit vectors. Returns R ∈ SO(3)."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    if w is None:
        w = np.ones(len(a))
    B = (w[:, None] * b).T @ a              # 3x3 attitude profile
    U, _, Vt = np.linalg.svd(B)
    d = np.linalg.det(U) * np.linalg.det(Vt)
    return U @ np.diag([1.0, 1.0, d]) @ Vt

a = rng.standard_normal((6, 3))
a /= np.linalg.norm(a, axis=1, keepdims=True)
b = a @ R_true.T
R_hat = wahba_svd(a, b)
print("Wahba       ‖R̂ - R‖ =", np.linalg.norm(R_hat - R_true))

## 3. Davenport's q-method and QUEST

Solves the *same* problem as Wahba's, but parameterises the rotation as a unit quaternion $q \in S^3$. This recasts the optimisation as an eigenvalue problem and avoids the SVD's $\det$ correction — the result lives natively on the quaternion sphere.

Build the same profile matrix $B = \sum_i w_i \mathbf{b}_i \mathbf{a}_i^\top$ and then

$$
S = B + B^\top, \qquad \sigma = \operatorname{tr} B, \qquad \mathbf{z} = \begin{bmatrix} B_{23} - B_{32} \\ B_{31} - B_{13} \\ B_{12} - B_{21} \end{bmatrix}.
$$

**Davenport's K matrix:**

$$
K \;=\; \begin{bmatrix} S - \sigma I_3 & \mathbf{z} \\ \mathbf{z}^\top & \sigma \end{bmatrix} \;\in\; \mathbb{R}^{4\times 4}.
$$

**q-method.** The optimal quaternion is the eigenvector corresponding to the *largest* eigenvalue:

$$
\boxed{\; K\, q^{\star} \;=\; \lambda_{\max}\, q^{\star}. \;}
$$

The minimum cost satisfies $J(R^{\star}) = \sum_i w_i - \lambda_{\max}$.

**QUEST.** Identical formulation, but instead of a full eigendecomposition QUEST solves the secular equation $\det(K - \lambda I) = 0$ for $\lambda_{\max}$ via Newton's method (1–2 iterations starting from $\lambda_0 = \sum_i w_i$), then recovers $q^{\star}$ from the linear system $(K - \lambda_{\max} I)\,q = 0$. This is what made it fast enough for 1980s flight computers.

**Where this shows up — flight-grade attitude estimation, spacecraft and UAV.** NASA Magsat (1979) onwards every spacecraft has used QUEST to convert star-tracker observations (5–10 catalog matches per frame at 10 Hz) into an attitude quaternion onboard — a full SVD or `eigh` is a luxury when you have 16 KB of RTOS memory and a hard real-time deadline. The same algorithm appears today inside PX4's EKF2 and ArduPilot's attitude initialiser: at boot, the autopilot uses the resting accelerometer (gravity) and magnetometer (B-field) as two Wahba observations and runs QUEST to seed the quaternion state of the EKF on a 168 MHz STM32F4. The output quaternion goes straight to the controller without ever materialising a $3\times 3$ rotation matrix.

In [ ]:
def davenport_q_method(a, b, w=None):
    """Optimal quaternion (qx, qy, qz, qw) for Wahba's problem."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    if w is None:
        w = np.ones(len(a))
    B = (w[:, None] * b).T @ a
    S = B + B.T
    sigma = np.trace(B)
    z = np.array([B[1, 2] - B[2, 1], B[2, 0] - B[0, 2], B[0, 1] - B[1, 0]])
    K = np.zeros((4, 4))
    K[:3, :3] = S - sigma * np.eye(3)
    K[:3,  3] = z
    K[ 3, :3] = z
    K[ 3,  3] = sigma
    eigvals, eigvecs = np.linalg.eigh(K)
    return eigvecs[:, np.argmax(eigvals)]

def quat_to_R(q):
    """Active rotation matrix from quaternion q = (qx, qy, qz, qw)."""
    qx, qy, qz, qw = q
    return np.array([
        [1 - 2*(qy*qy + qz*qz),   2*(qx*qy - qz*qw),   2*(qx*qz + qy*qw)],
        [    2*(qx*qy + qz*qw), 1 - 2*(qx*qx + qz*qz),   2*(qy*qz - qx*qw)],
        [    2*(qx*qz - qy*qw),   2*(qy*qz + qx*qw), 1 - 2*(qx*qx + qy*qy)],
    ])

q_hat = davenport_q_method(a, b)
# Davenport's K is derived in the *passive* (frame-rotation) convention,
# so the active rotation matrix is the transpose of quat_to_R(q).
R_hat = quat_to_R(q_hat).T
print("q-method    ‖R̂ - R‖ =", np.linalg.norm(R_hat - R_true))

## 4. Kabsch algorithm

**Problem.** Same math as orthogonal Procrustes restricted to $d = 3$ and to *rigid* (rotation + translation) transforms. Given correspondence-paired point clouds $P, Q \in \mathbb{R}^{n\times 3}$,

$$
(R^{\star}, \mathbf{t}^{\star}) \;=\; \arg\min_{R \in SO(3),\, \mathbf{t} \in \mathbb{R}^3} \sum_{i=1}^{n} \lVert \mathbf{q}_i - (R\,\mathbf{p}_i + \mathbf{t}) \rVert^2.
$$

**Solution.** The optimal translation is determined once $R$ is known: $\mathbf{t}^{\star} = \bar{\mathbf{q}} - R^{\star} \bar{\mathbf{p}}$. Substituting reduces the problem to centred Procrustes:

1. $\bar{\mathbf{p}} = \tfrac{1}{n}\sum \mathbf{p}_i,\quad \bar{\mathbf{q}} = \tfrac{1}{n}\sum \mathbf{q}_i$
2. $H = \tilde P^\top \tilde Q$ with $\tilde P = P - \bar{\mathbf{p}},\; \tilde Q = Q - \bar{\mathbf{q}}$
3. SVD $H = U \Sigma V^\top$
4. $d = \det(V U^\top)$, $\;D = \mathrm{diag}(1, 1, d)$
5. $\boxed{\; R^{\star} = V D U^\top, \qquad \mathbf{t}^{\star} = \bar{\mathbf{q}} - R^{\star} \bar{\mathbf{p}}. \;}$

The workhorse for any "rigid alignment with known correspondences" problem.

**Where this shows up — loop closure in RGB-D / stereo SLAM.** When the robot revisits a place, the front end (ORB, SIFT, SuperPoint, …) matches features between the current keyframe and a candidate past keyframe. Each match gives a 3D position in *both* keyframes — depth from the RGB-D sensor (Kinect, RealSense, iPhone LiDAR) or from stereo triangulation. **Kabsch on the matched 3D-3D pairs returns the relative pose between the two keyframes**, which becomes the loop-closure constraint added to the pose graph and then smoothed by g2o / GTSAM. The same routine is the rigid registration used in AprilTag-bundle calibration and in visual fiducial-based gripper / hand-eye sanity checks.

In [ ]:
def kabsch(P, Q):
    """P, Q: (N, 3) corresponding rows. Returns R ∈ SO(3), t with Q ≈ P @ R.T + t."""
    p_bar = P.mean(axis=0)
    q_bar = Q.mean(axis=0)
    H = (P - p_bar).T @ (Q - q_bar)
    U, _, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    D = np.diag([1.0, 1.0, d])
    R = Vt.T @ D @ U.T
    t = q_bar - R @ p_bar
    return R, t

P = rng.standard_normal((20, 3))
t_true = rng.standard_normal(3)
Q = P @ R_true.T + t_true
R_hat, t_hat = kabsch(P, Q)
print("Kabsch      ‖R̂ - R‖ =", np.linalg.norm(R_hat - R_true),
      "  ‖t̂ - t‖ =", np.linalg.norm(t_hat - t_true))

## 5. Umeyama algorithm

**Problem.** Extends Kabsch by also estimating a positive uniform scale $s$:

$$
(s^{\star}, R^{\star}, \mathbf{t}^{\star}) \;=\; \arg\min_{s>0,\, R \in SO(d),\, \mathbf{t}} \sum_{i=1}^{n} \lVert \mathbf{q}_i - (s\,R\,\mathbf{p}_i + \mathbf{t}) \rVert^2.
$$

**Solution.** Same centring trick as Kabsch, plus a scale step:

- $\Sigma_{pq} = \tfrac{1}{n} \sum_i (\mathbf{q}_i - \bar{\mathbf{q}})(\mathbf{p}_i - \bar{\mathbf{p}})^\top$
- $\sigma_p^2 = \tfrac{1}{n} \sum_i \lVert \mathbf{p}_i - \bar{\mathbf{p}} \rVert^2$
- SVD $\Sigma_{pq} = U D V^\top$
- $S = \mathrm{diag}\!\bigl(1, \ldots, 1, \operatorname{sign}(\det U \det V)\bigr)$
- $\boxed{\; R^{\star} = U S V^\top, \qquad s^{\star} = \frac{\operatorname{tr}(D S)}{\sigma_p^2}, \qquad \mathbf{t}^{\star} = \bar{\mathbf{q}} - s^{\star} R^{\star} \bar{\mathbf{p}}. \;}$

Used wherever a similarity transform appears: aligning monocular SLAM trajectories to ground truth (which fixes the scale ambiguity), aligning shapes between scans of unknown calibration, etc.

**Where this shows up — evaluating monocular SLAM on TUM-RGBD.** Monocular ORB-SLAM3 estimates the camera trajectory from a single video stream, but the *absolute scale* is fundamentally unobservable — a one-metre desk filmed up close is indistinguishable from a ten-metre desk filmed from far away. The estimated trajectory is therefore in arbitrary units. To compare it against the motion-capture ground truth from the TUM-RGBD benchmark you need rotation, translation, *and* an unknown scale. The widely-used `evo` evaluation toolkit calls `evo_ape ... -as` ("align with scale"), which is Umeyama under the hood, and reports the post-alignment trajectory error.

In [ ]:
def umeyama(P, Q):
    """Similarity transform (s, R, t) minimising Σ ‖qᵢ - (s R pᵢ + t)‖²."""
    n, d = P.shape
    p_bar = P.mean(axis=0)
    q_bar = Q.mean(axis=0)
    Pc = P - p_bar
    Qc = Q - q_bar
    Sigma_pq = (Qc.T @ Pc) / n
    sigma_p2 = (Pc * Pc).sum() / n
    U, D, Vt = np.linalg.svd(Sigma_pq)
    S = np.eye(d)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[-1, -1] = -1
    R = U @ S @ Vt
    s = (D * np.diag(S)).sum() / sigma_p2
    t = q_bar - s * R @ p_bar
    return s, R, t

s_true = 2.5
Q_scaled = s_true * (P @ R_true.T) + t_true
s_hat, R_hat, t_hat = umeyama(P, Q_scaled)
print(f"Umeyama     ŝ = {s_hat:.4f} (true {s_true})  "
      f"‖R̂ - R‖ = {np.linalg.norm(R_hat - R_true):.2e}  "
      f"‖t̂ - t‖ = {np.linalg.norm(t_hat - t_true):.2e}")

## 6. Iterative Closest Point (ICP)

**Problem.** Same goal as Kabsch — align $P$ onto $Q$ — but the correspondences are **not given**. The two clouds may even have different sizes ($n \ne m$).

$$
(R^{\star}, \mathbf{t}^{\star}) \;=\; \arg\min_{R,\, \mathbf{t}} \; \sum_{\mathbf{p}_i \in P} \min_{\mathbf{q}_j \in Q} \lVert \mathbf{q}_j - (R\,\mathbf{p}_i + \mathbf{t}) \rVert^2.
$$

**Algorithm.** Alternate two steps:

1. **Data association** — for each $\mathbf{p}_i$, find the closest $\mathbf{q}_j$ in the *current* aligned cloud (typically a kd-tree query).
2. **Transform refinement** — solve the resulting Procrustes/Kabsch problem on those correspondences, update $(R, \mathbf{t})$.

Repeat until $(R, \mathbf{t})$ stops changing (or the mean residual stabilises). Each iteration monotonically decreases the cost, but ICP only finds a **local minimum** — initialisation matters. Practical variants address this with point-to-plane cost, robust kernels (Huber, Tukey), normal-shooting, multi-resolution downsampling, etc.

ICP is the workhorse of LiDAR SLAM, depth-camera registration, and 3D scan stitching.

**Where this shows up — LiDAR odometry on a self-driving car.** A roof-mounted Velodyne or Ouster LiDAR captures a 3D scan of the surroundings every 100 ms. You don't know which point in scan $t+1$ corresponds to which point in scan $t$ — the car has moved, the laser has spun, points are sampled from different parts of nearby surfaces. ICP (or one of its many descendants — generalised ICP, point-to-plane, KISS-ICP) registers each new scan to a local map and reports the rigid motion between scans; integrate the per-frame transforms and you have **LiDAR odometry**, the backbone of the LOAM family of SLAM systems and most production AV stacks. The motion between consecutive scans is small enough that identity-initialisation (or the previous frame's velocity) reliably falls inside ICP's basin of convergence.

In [ ]:
from scipy.spatial import cKDTree

def icp(P, Q, max_iter=50, tol=1e-6):
    """Returns (R, t) aligning P onto Q. Identity initialisation."""
    tree = cKDTree(Q)
    R = np.eye(3)
    t = np.zeros(3)
    prev_err = np.inf
    for _ in range(max_iter):
        Pt = P @ R.T + t                  # current pose
        d, idx = tree.query(Pt)           # nearest target points
        R, t = kabsch(P, Q[idx])          # closed-form refinement
        err = (d ** 2).mean()
        if abs(prev_err - err) < tol:
            break
        prev_err = err
    return R, t

# ICP only converges from identity when the misalignment is small enough that
# initial nearest-neighbour matches are mostly correct. Use a ~6° rotation.
theta = 0.1
R_small = np.array([
    [np.cos(theta), -np.sin(theta), 0.0],
    [np.sin(theta),  np.cos(theta), 0.0],
    [0.0,            0.0,           1.0],
])
t_small = np.array([0.1, -0.05, 0.0])
Q_small = P @ R_small.T + t_small

# Shuffle to break the trivial 1:1 row correspondence.
perm = rng.permutation(len(Q_small))
R_hat, t_hat = icp(P, Q_small[perm])
print("ICP         ‖R̂ - R‖ =", np.linalg.norm(R_hat - R_small),
      "  ‖t̂ - t‖ =", np.linalg.norm(t_hat - t_small))

## Comparison

| Algorithm | Inputs | Output | Correspondences | Closed form? | Solution method | Domain |
|---|---|---|---|---|---|---|
| **Orthogonal Procrustes** | $A, B \in \mathbb{R}^{d \times n}$ | $R \in O(d)$ (or $SO(d)$) | known | yes | SVD of $B A^\top$ | general $d$-dim |
| **Wahba** | unit vectors $\{(\mathbf{a}_i, \mathbf{b}_i)\}$, weights $w_i$ | $R \in SO(3)$ | known | yes | SVD of profile + det correction | spacecraft / UAV attitude |
| **q-method / QUEST** | same as Wahba | quaternion $q \in S^3$ | known | yes (eig); QUEST iterates Newton on the secular equation | largest eigenvalue of Davenport's $4\times 4$ matrix $K$ | flight-grade onboard attitude estimator |
| **Kabsch** | $P, Q \in \mathbb{R}^{n \times 3}$ | $R \in SO(3),\ \mathbf{t}$ | known | yes | centred SVD | rigid 3D-3D registration, SLAM loop closure |
| **Umeyama** | $P, Q \in \mathbb{R}^{n \times d}$ | $s, R, \mathbf{t}$ (similarity) | known | yes | centred SVD + scale formula | monocular SLAM trajectory evaluation |
| **ICP** | $P, Q$ (sizes may differ) | $R, \mathbf{t}$ | **unknown — discovered iteratively** | no | nearest-neighbour + Kabsch per iteration | LiDAR / RGB-D scan registration |

### One concrete robotics application per algorithm

| Algorithm | Where you actually run into this |
|---|---|
| **Orthogonal Procrustes** | Re-orthonormalising a drifted 3×3 rotation matrix inside an EKF or pose integrator (`B = R, A = I` → SVD → nearest $SO(3)$); same trick repairs rotations regressed by a neural network |
| **Wahba** | Bootstrap attitude for a UAV / quadcopter on the ground: gravity (accelerometer) and Earth's B-field (magnetometer + WMM model) as the two reference vectors; same setup initialises a CubeSat from sun-sensor + magnetometer measurements |
| **q-method / QUEST** | Onboard attitude estimator on flight hardware — spacecraft star trackers since NASA Magsat (1979), and PX4/ArduPilot's quaternion EKF initialiser at boot on a 168 MHz STM32F4 |
| **Kabsch** | Loop closure in RGB-D / stereo SLAM — Kabsch on matched 3D-3D feature pairs gives the relative pose between two keyframes, which becomes a constraint added to the pose graph (g2o / GTSAM) |
| **Umeyama** | `evo_ape -as` aligning a monocular ORB-SLAM3 trajectory to motion-capture ground truth on TUM-RGBD / EuRoC (the `-as` flag = "with scale" — monocular has unobservable scale) |
| **ICP** | LiDAR odometry on a self-driving car or a quadruped: register each new Velodyne / Ouster / Livox scan to the previous one (or to a local map) at 10 Hz — the backbone of LOAM, KISS-ICP, FAST-LIO, and most production AV stacks |

**Bottom line.** Procrustes and Wahba are the same problem in different notation — Wahba is just Procrustes with the convention of "unit-vector observations" and the additional $SO(3)$ constraint. Kabsch is Procrustes lifted to point clouds with translation. Umeyama is Kabsch with a scale. q-method/QUEST swap the SVD for an eigenvalue formulation — same answer, different machinery. ICP wraps Kabsch in an outer loop to handle the case where you don't know which point matches which.

---

References:
- Schoenemann (1966), *A generalized solution of the orthogonal procrustes problem*.
- Wahba (1965), *A least squares estimate of satellite attitude*, SIAM Review.
- Shuster & Oh (1981), *Three-axis attitude determination from vector observations* (QUEST).
- Kabsch (1976), *A solution for the best rotation to relate two sets of vectors*, Acta Cryst.
- Umeyama (1991), *Least-squares estimation of transformation parameters between two point patterns*, IEEE PAMI.
- Besl & McKay (1992), *A method for registration of 3-D shapes*, IEEE PAMI (ICP).